# Setup

In [1]:
!pip -qq install langchain Groq langchain_core langchain_groq streamlit pyswip mdextractor

# SWI CrossJustice

In [2]:
%%writefile swi_interface.py
import re
import json
import os
import pyswip, ctypes

#root = os.path.abspath(os.path.join(os.path.dirname(__file__), os.pardir))
root = "/workspaces/crossjustice-bot/src"

class PrologMT(pyswip.Prolog):
    """Multi-threaded (one-to-one) pyswip.Prolog ad-hoc reimpl"""

    _swipl = pyswip.core._lib

    PL_thread_self = _swipl.PL_thread_self
    PL_thread_self.restype = ctypes.c_int

    PL_thread_attach_engine = _swipl.PL_thread_attach_engine
    PL_thread_attach_engine.argtypes = [ctypes.c_void_p]
    PL_thread_attach_engine.restype = ctypes.c_int

    @classmethod
    def _init_prolog_thread(cls):
        pengine_id = cls.PL_thread_self()
        if pengine_id == -1:
            pengine_id = cls.PL_thread_attach_engine(None)
            print("{INFO} attach pengine to thread: %d" % pengine_id)
        if pengine_id == -1:
            raise pyswip.prolog.PrologError(
                "Unable to attach new Prolog engine to the thread"
            )
        elif pengine_id == -2:
            print("{WARN} Single-threaded swipl build, beware!")

    class _QueryWrapper(pyswip.Prolog._QueryWrapper):
        def __call__(self, *args, **kwargs):
            PrologMT._init_prolog_thread()
            return super().__call__(*args, **kwargs)


def source_path(directive, implementation):
    dict = {
        "pl": "polish_implementation",
        "it": "italian_implementation",
        "es": "spanish_implementation",
        "nl": "dutch_implementation",
        "bg": "bulgarian_implementation",
        "dir": directive,
        "directive": directive,
    }

    return os.path.join(
        root, f"prolog/directives/{directive}/{dict[implementation]}.pl"
    ).replace("\\", "/")


def nationality(law):
    element = {
        "_it": "it",
        "_pl": "pl",
        "_nl": "nl",
        "_es": "es",
        "_bg": "bg",
        "": "directive",
    }
    return next((y for x, y in element.items() if x in law))


def _generate_interpreter():
    def _add_directive(swipl, root, directive, implementations):
        def _impl(directive, implementation):

            module = source_path(directive, implementation)
            next(swipl.query(f"use_module('{module}',[])"))

            return (
                implementation,
                directive
                if implementation == "dir"
                else directive + "_" + implementation,
            )

        return [_impl(directive, "dir")] + [
            _impl(directive, x) for x in implementations
        ]

    swipl = PrologMT()
    # root = os.path.abspath(os.path.join(os.path.dirname(__file__), os.pardir))
    swipl.consult(os.path.join(root, "prolog/my_init.pl").replace("\\", "/"))

    return (
        swipl,
        _add_directive(swipl, root, "directive_2010_64", ["it", "es", "nl", "pl", "bg"])
        + _add_directive(swipl, root, "directive_2016_800", ["it", "nl", "pl", "bg"])
        + _add_directive(swipl, root, "directive_2012_13", ["it", "nl", "pl", "bg"])
        + _add_directive(swipl, root, "directive_2016_343", ["it", "nl", "pl", "bg"])
        + _add_directive(swipl, root, "directive_2013_48", ["it", "nl", "pl", "bg"])
        + _add_directive(swipl, root, "directive_2016_1919", ["it", "nl", "pl", "bg"]),
    )


_swipl, _mappings = _generate_interpreter()


def _json_clean_explanation(expl):
    def _clean_explanation(tmp_list):
        log = []
        for tmp in tmp_list:
            if isinstance(tmp, list):
                log.append(_clean_explanation(tmp))
            else:
                tmp = re.sub("Variable\(\d+\)", "_", str(tmp))
                if (
                    tmp != "system_predicate"
                    and not tmp.startswith("auxiliary_right_scope")
                    and not tmp.startswith("right_property_scope")
                ):
                    log.append(tmp)
        return log

    return json.dumps(_clean_explanation(expl))


def _with_facts(execute, facts, module):
    for item in facts:
        _swipl.assertz(module + ":" + item)

    result = execute()

    for item in facts:
        _swipl.retractall(module + ":" + item)

    return result


def _solve(query):
    return list(_swipl.query(query))


def _with_explanation(module, query, explanation=True):
    return f"{module}:explain({query}, Expl)" if explanation else f"{module}:{query}"


def right_to(
    person,
    facts,
    laws=["it", "es", "pl", "nl", "bg", "dir"],
    modules=[],
    right="Right",
    explanation=False,
):

    return set(
        (
            law,
            module,
            x["Art"],
            x["Opt"],
            x["Right"] if right == "Right" else right,
            _json_clean_explanation(x["Expl"]) if explanation else "",
        )
        for law, module in _mappings
        if law in laws or module in modules
        for x in _with_facts(
            lambda: _solve(
                _with_explanation(
                    module,
                    f"has_right({right}, {law}, Art, {person}, Opt)",
                    explanation,
                )
            ),
            facts,
            module,
        )
    )


def auxiliary_right(
    person,
    facts,
    laws=["it", "es", "pl", "nl", "bg", "dir"],
    modules=[],
    ref="Ref",
    checked=True,
    explanation=False,
):

    return set(
        (
            law,
            module,
            x["Art"],
            x["Opt"],
            x["Value"],
            x["Ref"] if ref == "Ref" else ref,
            _json_clean_explanation(x["Expl"]) if explanation else "",
        )
        for law, module in _mappings
        if law in laws or module in modules
        for x in _with_facts(
            lambda: _solve(
                _with_explanation(
                    module,
                    f"auxiliary_right{ '_checked' if checked else '' }(Art, {ref}, {person}, Opt, Value)",
                    explanation,
                )
            ),
            facts,
            module,
        )
    )


def right_property(
    person,
    facts,
    laws=["it", "es", "pl", "nl", "dir"],
    modules=[],
    ref="Ref",
    checked=True,
    explanation=False,
):

    return set(
        (
            law,
            module,
            x["Art"],
            x["Opt"],
            x["Value"],
            x["Ref"] if ref == "Ref" else ref,
            _json_clean_explanation(x["Expl"]) if explanation else "",
        )
        for law, module in _mappings
        if law in laws or module in modules
        for x in _with_facts(
            lambda: _solve(
                _with_explanation(
                    module,
                    f"right_property{ '_checked' if checked else '' }(Art, {ref}, {person}, Opt, Value)",
                    explanation,
                )
            ),
            facts,
            module,
        )
    )

def generate_response(personId, facts, laws):

    return facts, [
        {
            "right": item[4],
            "law": item[1],
            "article": item[2],
            "option": item[3],
            "explanation": item[5] if not item[5] else json.loads(item[5]),
            "auxiliary": [],
            "property": [],
            # "auxiliary": [
            #     {
            #         "article": aux[2],
            #         "matter": aux[3],
            #         "value": aux[4],
            #         "explanation": aux[6] if not aux[6] else json.loads(aux[6]),
            #     }
            #     for aux in auxiliary_right(
            #         form.base_info.defendant_name(),
            #         facts,
            #         laws=[],
            #         modules=[item[1]],
            #         ref=item[2],
            #         checked=False,
            #         explanation=form.explanation(),
            #     )
            # ],
            # "property": [
            #     {
            #         "article": aux[2],
            #         "matter": aux[3],
            #         "value": aux[4],
            #         "explanation": aux[6] if not aux[6] else json.loads(aux[6]),
            #     }
            #     for aux in right_property(
            #         form.base_info.defendant_name(),
            #         facts,
            #         laws=[],
            #         modules=[item[1]],
            #         ref=item[2],
            #         checked=False,
            #         explanation=form.explanation(),
            #     )
            # ],
        }
        for item in right_to(
            personId,
            facts,
            laws=laws,
            modules=[],
            right="Right",
            explanation=True,
        )
    ]

def prettify_response(response, facts):

    def _get_article_text(type, law, article):
        directive = law[:-3] if not law[-1].isdigit() else law
        path = source_path(directive, nationality(law))
        with open(path, "r") as pl_src:
            law_text = pl_src.read()
            # res = re.search(r'has_right\(_' + article + '.+\n%\n%([^()]+)\n+', law_text)
            res = re.search(
                type
                + r"\(_"
                + article
                + ".+\n%\n%\s?([^%]+?(?=%))([\S\s%\n]+?)(?=\n[a-z]|\$)(\$[\s\S]*?\$)?",
                law_text,
            )
            if res is not None:
                title = res.group(1).replace("%", "")
                text = res.group(2).replace("%", " ")
                comment = (res.group(3) or "").replace("%", " ").replace("$", "")
                return [title, text, comment]
        return ""

    def _to_html_tree(tree, facts):
        if tree == "":
            return ""
        log = "<ul>"
        for x in tree:
            if isinstance(x, list) and not x:
                continue
            if isinstance(x, list):
                log += _to_html_tree(x, facts)
            elif x.startswith(":(user"):
                log += (
                    "<li>"
                    + re.search(r":\(user, (.+)\)", x).group(1)
                    + " <b>[FACT]</b></li>"
                )
            elif x in facts:
                log += "<li>" + x + " <b>[FACT]</b></li>"
            else:
                log += "<li>" + x + "</li>"
        log += "</ul>"
        return log

    def _prettify(elem, type, law):
        elem["article_text"] = _get_article_text(type, law, elem["article"])
        elem["explanation_raw"] = elem["explanation"]
        elem["explanation"] = _to_html_tree(elem["explanation"], facts)
        return elem

    def _extract_facts(explanation):
        if explanation == "":
            return []
        temp = []
        for x in explanation:
            if isinstance(x, list) and not x:
                continue
            if isinstance(x, list):
                temp += _extract_facts(x)
            if x in facts:
                temp += [x]
        return temp

    def _to_theory(elem):

        for law, rights in elem.items():
            if law == "directive":
                for y in rights:
                    y[
                        "arg_rule"
                    ] = f"right({law}, {y['article']}, {y['right']}, {y['option']}, {sorted(set(_extract_facts(y['explanation_raw'])))}).\\n".replace(
                        "'", ""
                    )
                    for prop in y["auxiliary"]:
                        prop[
                            "arg_rule"
                        ] = f"auxiliary({law}, {prop['article']}, {prop['matter']}, {prop['value']}, {sorted(set(_extract_facts(prop['explanation_raw'])))}).\\n".replace(
                            "'", ""
                        )
                    for prop in y["property"]:
                        prop[
                            "arg_rule"
                        ] = f"property({law}, {prop['article']}, {prop['matter']}, {prop['value']}, {sorted(set(_extract_facts(prop['explanation_raw'])))}).\\n".replace(
                            "'", ""
                        )

        return (
            "\\n".join(
                [
                    f":= (right({law}, {y['article']}, {y['right']}, {y['option']}, {sorted(set(_extract_facts(y['explanation_raw'])))}))."
                    for law, rights in elem.items()
                    for y in rights
                    if law != "directive"
                ]
            ).replace("'", ""),
            "\\n".join(
                [
                    f":= (auxiliary({law}, {prop['article']}, {prop['matter']}, {prop['value']}, {sorted(set(_extract_facts(prop['explanation_raw'])))})."
                    for law, rights in elem.items()
                    for y in rights
                    for prop in y["auxiliary"]
                    if law != "directive"
                ]
            ).replace("'", ""),
            "\\n".join(
                [
                    f":= (property({law}, {prop['article']}, {prop['matter']}, {prop['value']}, {sorted(set(_extract_facts(prop['explanation_raw'])))})."
                    for law, rights in elem.items()
                    for y in rights
                    for prop in y["property"]
                    if law != "directive"
                ]
            ).replace("'", ""),
        )

    import copy

    pretty = [copy.deepcopy(dict) for dict in response]

    for elem in pretty:
        _prettify(elem, "has_right", elem["law"])
        elem["property"] = [
            _prettify(property, "right_property", elem["law"])
            for property in elem["property"]
        ]
        elem["auxiliary"] = [
            _prettify(property, "auxiliary_right", elem["law"])
            for property in elem["auxiliary"]
        ]

    return pretty

    # group = {}
    # for item in pretty:
    #     group.setdefault(item["right"], []).append(item)
    # for key, value in group.items():
    #     national_group = {}
    #     for item in value:
    #         national_group.setdefault(nationality(item["law"]), []).append(item)
    #     group[key] = (national_group, *_to_theory(national_group))

    # return group

Overwriting swi_interface.py


In [3]:
def test_facts():
    return [
        "proceeding_type(personId, criminal)",
        "person_status(personId, accused)",
        "person_made_aware(personId, personStatus)",
        "proceeding_status(personId, started)",
        "person_understands(personId, unknown)",
        "person_speaks(personId, unknown)",
        "proceeding_language(personId, langUnknown)"
    ]

In [4]:
#facts, response = generate_response("personId", test_facts(), ["dir"])
#pretty = prettify_response(response, facts)

#print(response)

In [5]:
#pretty[0]

# Facts

In [6]:
extra = """
%% person_reads(PersonId, Language)
%
% The person speaks, reads and understands language X
%

person_reads(PersonId, Language).
"""

In [7]:
%%writefile input_facts.py
facts = """
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

% GENERAL FACTS

%% proceeding_type(PersonId, _criminal)
%
% The proceedings are of criminal nature
%

proceeding_type(PersonId, criminal).


%% proceeding_type(PersonId, _europeanArrestWarrant)
%
% The proceedings concern the execution of a european arrest warrant
%

proceeding_type(PersonId, europeanArrestWarrant).


%% person_status(PersonId, _suspect)
%
% The person is a suspect
%

person_status(PersonId, suspect).


%% person_status(PersonId, _accused)
%
% The person is an accused
%

person_status(PersonId, accused).


%% person_made_aware(PersonId, _personStatus)
%
% The person has been made aware of his status
%

person_made_aware(PersonId, personStatus).


%% proceeding_status(PersonId, _started)
%
% The proceedings have started
%

proceeding_status(PersonId, started).


%% proceeding_status(PersonId, _concluded)
%
% The proceedings have reached the final determination
%

proceeding_status(PersonId, concluded).
"""

facts_dir_2010_64 = """
Directive 2010_64
%% person_document(PersonId, _europeanArrestWarrant)
%
% The document issued is the european arrest warrant
%

person_document(PersonId, europeanArrestWarrant).


%% authority_decision(PersonId, _essential_document)
%
% The authority has decreed any document to be essential
%

authority_decision(PersonId, essential_document).


%% person_request_submitted(PersonId, _essential_document)
%
% The person has submitted a request to consider a document to be essential
%

person_request_submitted(PersonId, essential_document).


%% person_document(PersonId, _document_deprives_liberty)
%
% The document issued is any document that deprives liberty
%

person_document(PersonId, document_deprives_liberty).


%% person_document(PersonId, _charge)
%
% The document issued is the charge
%

person_document(PersonId, charge).


%% person_document(PersonId, _indictment)
%
% The document issued is an indictment
%

person_document(PersonId, indictment).


%% person_document(PersonId, _judgement)
%
% The document issued is a judgement
%

person_document(PersonId, judgement).


%% person_understands(PersonId, Language)
%
% The person reads and understands language X
%

person_understands(PersonId, Language).

%% lawyer_language(PersonId, Language)
%
% The lawyer can speak, read and understand this language
%

lawyer_language(PersonId, Language).

%% person_language(PersonId, Language)
%
% The person speaks, reads and understands language X
%

person_language(PersonId, Language).

%% proceeding_language(PersonId, Language)
%
% The language of the proceedings is X
%

proceeding_language(PersonId, Language).
"""

facts_dir_2016_800 = """
% Directive 2016/800

%% person_status(PersonId, _child)
%
% The person is a child
%

person_status(PersonId, child).


%% person_status(PersonId, _subject_to_proceedings)
%
% The person is the subject of the proceedings
%

person_status(PersonId, subject_to_proceedings).


%% person_age(PersonId, Age, _start_proceeding)
%
% The person was X years of age at the start of the proceedings
%

person_age(PersonId, Age, start_proceeding).


%% person_age(PersonId, Age)
%
% The person is currently X years of age
%

person_age(PersonId, Age).


%% person_status(PersonId, _deprived_of_liberty)
%
% The person has been deprived of liberty
%

person_status(PersonId, deprived_of_liberty).


%% person_nature(PersonId, _natural)
%
% The person is a natural person
%

person_nature(PersonId, natural).


%% person_nature(PersonId, _legal)
%
% The person is a legal person
%

person_nature(PersonId, legal).


%% holds_parental_responsibility(PersonId, ChildId)
%
% This person holds parental responsibility over the child
%

holds_parental_responsibility(HolderId, PersonId).



%% responsibility_assigned(PersonId, ChildId, _judgement)
%
% The responsiblity over the child has been assigned through a judgement
%

responsibility_assigned(PersonId, ChildId, judgement).


%% responsibility_assigned(PersonId, ChildId, _operation_of_law)
%
% The responsiblity over the child has been assigned through an operation of law
%

responsibility_assigned(PersonId, ChildId, operation_of_law).


%% responsibility_assigned(PersonId, ChildId, _agreement_having_legal_effects)
%
% The responsiblity over the child has been assigned through an agreement having legal effect
%

responsibility_assigned(PersonId, ChildId, agreement_having_legal_effects).


%% person_status(PersonId, _age_unknown)
%
% The person's age is unknown
%

person_status(PersonId, age_unknown).


%% proceeding_matter(PersonId, _court_hearing)
%
% The proceedings concern the court hearing
%

proceeding_matter(PersonId, court_hearing).


%% proceeding_matter(PersonId, _questioning)
%
% The proceedings concern the questioning of the defendant
%

proceeding_matter(PersonId, questioning).


%% proceeding_matter(PersonId, _evidence_gathering_act)
%
% The proceedings concern any evidence gathering act
%

proceeding_matter(PersonId, evidence_gathering_act).


%% proceeding_matter(PersonId, _summoned_court)
%
% The proceedings concern the summon to court of the person
%

proceeding_matter(PersonId, summoned_court).


%% proceeding_matter(PersonId, _identity_parade)
%
% The proceedings concern an identity parade
%

proceeding_matter(PersonId, identity_parade).


%% proceeding_matter(PersonId, _confrontations)
%
% The proceedings concern confrontation
%

proceeding_matter(PersonId, confrontations).


%% proceeding_matter(PersonId, _reconstructions_crime_scene)
%
% The proceedings concern the reconstructions of a crime scene
%

proceeding_matter(PersonId, reconstructions_crime_scene).

%% holder_nomination(ChildId, HolderId)
%
% The child has nominated someone as holder of parental responsibility
%

holder_nomination(ChildId, HolderId).


%% authority_decision(PersonId, _accepted)
%
% The authority has accepted the nomination of the child concerning the holder of parental responsibility
%

authority_decision(PersonId, accepted).


%% authority_nominates(PersonId, HolderId)
%
% The authority has nominated someone to accompany the child as responsibility holder
%

authority_nominates(PersonId, HolderId).


%% person_status(HolderId, _holder_of_parental_responsibility)
%
% The person is an holder of parental responsibility
%

person_status(HolderId, holder_of_parental_responsibility).


%% contrary_to_child_interest(PersonId)
%
% The person is acting contrary to the interests of the child
%

contrary_to_child_interest(PersonId).


%% jeopardises_proceedings(HolderId)
%
% This person's acting jeopardises the proceedings
%

jeopardises_proceedings(HolderId).

%% authority_decision(HolderId, _nomination)
%
% The authority has nominated a new holder of parental responsibility
%

authority_decision(HolderId, nomination).


%% proceeding_matter(PersonId, _detention)
%
% The proceedings concern detention
%

proceeding_matter(PersonId, detention).


%% person_status(PersonId, _detained)
%
% The person has been detained
%

person_status(PersonId, detained).


%% person_status(PersonId, _specific_health_indications)
%
% The person is in need of specific health indications
%

person_status(PersonId, specific_health_indications).


%% person_request_submitted(PersonId, _medical_exam)
%
% The person has submitted a request to perform a medical exam
%

person_request_submitted(PersonId, medical_exam).


%% person_request_submitted(HolderId, _medical_exam)
%
% The holder of parental responsibility has submitted a request to perform a medical exam
%

person_request_submitted(HolderId, medical_exam).


%% person_request_submitted(PersonId, _lawyer_medical_exam)
%
% The person's lawyer has submitted a request to perform a medical exam
%

person_request_submitted(PersonId, lawyer_medical_exam).


%% person_danger(PersonId, _life)
%
% There is need to avoid damage to a person's life
%

person_danger(PersonId, life).


%% person_danger(PersonId, _liberty)
%
% There is need to avoid damage to a person's liberty
%

person_danger(PersonId, liberty).


%% person_danger(PersonId, _physical_integrity)
%
% There is need to avoid damage to a person's physical integrity
%

person_danger(PersonId, physical_integrity).


%% proceeding_matter(PersonId, _pre_trial)
%
% The proceedings are happening before the trial has started
%

proceeding_matter(PersonId, pre_trial).


%% imperative_immediate_action(PersonId, _investigating_authority)
%
% Investigating authority must intervene to avoid jeopardising the proceedings
%

imperative_immediate_action(PersonId, investigating_authority).


%% proceeding_matter(PersonId, _trial_in_absentia)
%
% The proceedings have been conducted in absentia
%

proceeding_matter(PersonId, trial_in_absentia).
"""

Overwriting input_facts.py


# App

In [39]:
%%writefile app.py
import streamlit as st
from groq import Groq
import random

from langchain.chains import ConversationChain, LLMChain
from langchain_core.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.messages import SystemMessage
from langchain.chains.conversation.memory import ConversationBufferWindowMemory
from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate

from swi_interface import generate_response, prettify_response
import input_facts

def prompt_model(prompt, system="", model="llama3-70b-8192", temperature=0.5):

  client = Groq(
      api_key='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',
  )

  completion = client.chat.completions.create(
      model=model,
      messages=[
          {
              "role": "system",
              "content": system
          },
          {
              "role": "user",
              "content": prompt
          }
      ],
      temperature=temperature,
      top_p=1,
      stream=True,
      stop=None,
  )

  return "".join([chunk.choices[0].delta.content or "" for chunk in completion])


person_id = "anonym"

def parse_output(user_text):

  def extract_code(completion):
    from mdextractor import extract_md_blocks
    blocks = extract_md_blocks(completion)
    return blocks[0] if blocks else ""

  user_facts = [x for x in extract_code(user_text).replace(".", "").split("\n") if x]
  #if user_facts:
    #personId = "anonym"
    #personId = prompt(f"Extract the personId from this list of facts: {user_facts}.")

  return user_text, user_facts

def llm_response(input_text, person_id, user_facts, rights, memory):

  def select_prompt():

    res = prompt_model(f"According to the following input, does the user want to know her rights?\nReply OUTPUT input if that is the case.\n\n{input_text}")
    print(res)

    return f"""
        You must help the user to describe their case and then extract the relevant facts from this list of facts:

        {input_facts.facts + input_facts.facts_dir_2016_800}

        Make sure that all the facts are extracted correctly!
        Do not create new facts!

        Include a list of Prolog facts extracted so far inside a markdown block.

        Suggest to the user facts that could be relevant but he didnt mention.

        Do not put textual comments inside the Prolog markdown block.
        All variables must be grounded for a fact to be valid.
        If you dont know the value of PersonId, use anonym.
      """ if "INPUT" in res or not rights else f"""

        These are the rights computed by the symbolic engine til now:

        {rights[0]}

        Make the user know them.
      """

  groq_chat = ChatGroq(
      groq_api_key='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',
      model_name="llama3-70b-8192"
      #model_name="mixtral-8x7b-32768"
  )

  prompt = ChatPromptTemplate.from_messages(
        [
            SystemMessage(content=select_prompt()),
            MessagesPlaceholder(variable_name="chat_message"),
            HumanMessagePromptTemplate.from_template("{human_input}")
        ])

  conversation = LLMChain(
      llm=groq_chat,
      prompt=prompt,
      verbose=False,
      memory=memory,
  )

  return conversation.predict(human_input=input_text)

def main():

    st.title("Chat with CrossJustice!")

    if "chat_message" not in st.session_state:
      st.session_state.facts = []
      st.session_state.rights = []
      st.session_state.chat_message = []
      st.session_state.memory = ConversationBufferWindowMemory(k=10, memory_key="chat_message", return_messages=True)

    for message in st.session_state.chat_message:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

    if user_question := st.chat_input("Describe your case:"):

        with st.chat_message("user"):
          st.markdown(user_question)

        st.session_state.chat_message.append({"role": "user", "content": user_question})

        response = llm_response(user_question, person_id, st.session_state.facts, st.session_state.rights, st.session_state.memory)

        try:
          txt, user_facts = parse_output(response)
          if user_facts:
            facts, prolog_response = generate_response(person_id, user_facts, ["dir"])
            pretty = prettify_response(prolog_response, facts)
            st.session_state.rights = pretty
            st.session_state.facts = user_facts
            print(facts)
            print(pretty)
        except Exception as e:
          print(e)

        with st.chat_message("ai"):
          st.markdown(response)

        st.session_state.chat_message.append({"role": "ai", "content": response})

if __name__ == "__main__":
    main()

Overwriting app.py


# Start me

In [41]:
!streamlit run app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://10.0.15.174:8501
  External URL: http://20.61.127.62:8501

/workspaces/crossjustice-bot/src/app.py:123: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  st.session_state.memory = ConversationBufferWindowMemory(k=10, memory_key="chat_message", return_messages=True)
OUTPUT: Yes, the user wants to know her rights.

Although the user doesn't explicitly ask about their rights, the context of the situation (being investigated in a foreign country, not speaking the language, and being a minor) implies that they may be seeking information about their rights and protections in this situation.
/workspaces/crossjustice-bot/src/app.py:106: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` inste